# Part 3.2 — DAWNBench Challenge: ResNet-18 on CIFAR-10 (from scratch)

ResNet-18 built from first principles in PyTorch — no `torchvision.models`, no pretrained
weights. Trained on CIFAR-10 (60,000 32x32 colour images, 10 classes: 50,000 train / 10,000 test).

**Targets**

| goal | target |
|---|---|
| required | > 90% test accuracy, < ~30 min on GPU |
| stretch (DAWNBench) | ~94% accuracy in ~360 s on an A100, using mixed precision |

**Staged structure** — each stage is a separate cell you can run or skip independently:

| stage | what it does | cost |
|---|---|---|
| 1 | Architecture sanity check — random noise through the model, check output shape | seconds, CPU |
| 2 | Tiny smoke test — real data, 2 batches x 2 epochs, confirm loss drops, no NaN | ~1 min |
| 3 | Full training run, FP32, timed | minutes on GPU |
| 4 | Same run with mixed precision (AMP), timed — compare against stage 3 | minutes on GPU |

Stages 3 and 4 are gated behind environment variables so a quick smoke-test job on the
cluster can run stages 1-2 and stop. See the config cell below.

## Setup and configuration

Everything tunable is read from environment variables so the *same* notebook can be run
non-interactively under SLURM with different settings, e.g.

```bash
DAWN_EPOCHS=1 DAWN_STAGE3=1 jupyter nbconvert --execute --to notebook part3_dawnbench.ipynb
```

Defaults are chosen so that just opening and running the notebook does stages 1-2 only.

In [ ]:
import os
import time
import json
import platform

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset

import torchvision
import torchvision.transforms as transforms

import matplotlib
matplotlib.use("Agg")           # non-interactive backend: figures save to disk under nbconvert
import matplotlib.pyplot as plt


def env_int(name, default):
    """Read an int from the environment, falling back to `default` if unset/blank."""
    raw = os.environ.get(name, "").strip()
    return int(raw) if raw else default


def env_flag(name, default=False):
    """Read a 0/1 style flag from the environment."""
    raw = os.environ.get(name, "").strip().lower()
    if not raw:
        return default
    return raw in ("1", "true", "yes", "y", "on")


# ---------------------------------------------------------------- device
# cuDNN benchmark autotunes convolution algorithms for our fixed 32x32 input size.
# It costs a few seconds on the first batch and pays that back on every later batch.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    torch.backends.cudnn.benchmark = True

# ---------------------------------------------------------------- reproducibility
SEED = env_int("DAWN_SEED", 42)
torch.manual_seed(SEED)
np.random.seed(SEED)
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)

# ---------------------------------------------------------------- hyperparameters
DATA_DIR = os.environ.get("DAWN_DATA", "./data")   # where CIFAR-10 lives / is downloaded to
DOWNLOAD = env_flag("DAWN_DOWNLOAD", True)         # set 0 on compute nodes with no internet

EPOCHS = env_int("DAWN_EPOCHS", 30)                # 30 epochs of OneCycle reaches ~94%
BATCH_SIZE = env_int("DAWN_BATCH", 128)
MAX_LR = float(os.environ.get("DAWN_MAX_LR", "0.1"))
WEIGHT_DECAY = 5e-4
LABEL_SMOOTHING = 0.1

# DataLoader workers. 0 on Windows (notebook + multiprocessing spawn is fragile there);
# 4 on Linux, which is what the cluster runs.
NUM_WORKERS = env_int("DAWN_WORKERS", 0 if platform.system() == "Windows" else 4)

# Stage gates — stages 3 and 4 are expensive, so they are opt-in.
RUN_STAGE3 = env_flag("DAWN_STAGE3", False)        # full FP32 training run
RUN_STAGE4 = env_flag("DAWN_STAGE4", False)        # full mixed-precision training run

print(f"device       : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU          : {torch.cuda.get_device_name(0)}")
print(f"torch        : {torch.__version__}")
print(f"torchvision  : {torchvision.__version__}")
print(f"epochs       : {EPOCHS}")
print(f"batch size   : {BATCH_SIZE}")
print(f"max lr       : {MAX_LR}")
print(f"workers      : {NUM_WORKERS}")
print(f"data dir     : {DATA_DIR}")
print(f"run stage 3  : {RUN_STAGE3}")
print(f"run stage 4  : {RUN_STAGE4}")

## The architecture

### Why a residual block?

Stacking plain conv layers past ~20 layers makes networks *harder* to train, not easier —
gradients degrade on the way back and accuracy saturates then drops. A residual block sidesteps
this by learning a **correction** to its input rather than a whole new representation:

```
  x ──┬─────────────────────────────────────────────┐
      │                                             │
      └──> conv3x3 -> BN -> ReLU -> conv3x3 -> BN ──(+)──> ReLU ──> out
                       F(x), the "residual"          ▲
                                               identity x
```

The output is `relu(F(x) + x)`. If the ideal thing for this block to do is nothing, it only has
to drive `F(x)` towards zero, which is easy. And because the `+ x` branch is a plain addition,
gradients flow back through it unchanged — that is what makes 18+ layers trainable.

### The shape problem, and the shortcut

When a block changes the number of channels (64 -> 128) or halves the spatial size (stride 2),
`F(x)` and `x` no longer have matching shapes and cannot be added. The fix is a **projection
shortcut**: a 1x1 conv (plus BatchNorm) on the identity branch, with the same stride, purely to
reshape it. Blocks that keep the shape use the identity untouched.

### ResNet-18 layout

18 weighted layers = 1 stem conv + (2 blocks x 2 convs) x 4 stages + 1 fully-connected.

| stage | blocks | channels | stride | feature map |
|---|---|---|---|---|
| stem | — | 64 | 1 | 32x32 |
| layer1 | 2 | 64 | 1 | 32x32 |
| layer2 | 2 | 128 | 2 | 16x16 |
| layer3 | 2 | 256 | 2 | 8x8 |
| layer4 | 2 | 512 | 2 | 4x4 |
| head | — | — | — | avgpool -> 512 -> fc -> 10 |

### One important deviation from the paper

The original ImageNet ResNet-18 starts with a **7x7 stride-2 conv followed by a 3x3 stride-2
max-pool**, which shrinks a 224x224 input to 56x56 before the first block. Applying that to a
32x32 CIFAR image would leave 8x8 immediately and throw away most of the spatial detail —
accuracy caps out well below 90%.

So we use the standard **CIFAR stem**: a single 3x3 stride-1 conv, no max-pool, keeping the full
32x32 resolution into layer1. This is the usual adaptation for CIFAR-sized inputs and is the
single most important change for hitting the accuracy target.

In [ ]:
class BasicBlock(nn.Module):
    """The standard ResNet residual block: conv-BN-ReLU-conv-BN, then add the input back.

    Args:
        in_channels:  channels arriving at this block
        out_channels: channels this block produces
        stride:       stride of the FIRST conv. 2 halves the feature map, 1 keeps it.

    Shape: (B, in_channels, H, W) -> (B, out_channels, H // stride, W // stride)
    """

    # BasicBlock keeps the channel count it is given. (The deeper ResNet-50+ "Bottleneck"
    # block multiplies it by 4; this attribute is what lets the same ResNet class drive both.)
    expansion = 1

    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()

        # --- the residual branch F(x) ---------------------------------------------------
        # bias=False on every conv that is followed by BatchNorm: BN subtracts the batch mean
        # and then adds its own learnable shift beta, so a conv bias would be immediately
        # cancelled out. Dropping it saves parameters and does not change what the net can learn.
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3,
                               stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)

        # Second conv always has stride 1 — the block downsamples at most once, in conv1.
        # padding=1 with a 3x3 kernel keeps the spatial size unchanged ("same" padding).
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3,
                               stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        # --- the identity branch --------------------------------------------------------
        # If the block preserves both shape and channel count, the shortcut is literally
        # "do nothing" — an empty Sequential is the identity function.
        self.shortcut = nn.Sequential()

        # Otherwise we need a projection so the two branches can be added. A 1x1 conv with
        # the same stride fixes channels and spatial size in one step.
        if stride != 1 or in_channels != out_channels * self.expansion:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels * self.expansion,
                          kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels * self.expansion),
            )

    def forward(self, x):
        # Compute the identity branch first — it reads the ORIGINAL x, not the transformed one.
        identity = self.shortcut(x)

        # Residual branch: conv -> BN -> ReLU -> conv -> BN.
        # Note there is deliberately NO ReLU after bn2 yet.
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))

        # The skip connection: add the block's input back in.
        out = out + identity

        # The final ReLU comes AFTER the addition. This ordering matters — it is what keeps
        # the identity path a clean, unsquashed route for gradients to flow backwards through.
        return F.relu(out)

In [ ]:
class ResNet(nn.Module):
    """ResNet for 32x32 inputs, built from residual blocks.

    Args:
        block:      block class to stack (BasicBlock here)
        num_blocks: how many blocks in each of the 4 stages, e.g. [2, 2, 2, 2] for ResNet-18
        num_classes: size of the output layer (10 for CIFAR-10)
    """

    def __init__(self, block, num_blocks, num_classes=10):
        super().__init__()

        # Tracks how many channels the next block will receive. _make_layer updates it.
        self.in_channels = 64

        # --- CIFAR stem -----------------------------------------------------------------
        # 3x3 stride-1, no max-pool: preserves the full 32x32 resolution. (See the note above
        # on why we do not use the ImageNet 7x7 stride-2 + maxpool stem.)
        # 3 input channels because CIFAR-10 images are RGB.
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)

        # --- four residual stages -------------------------------------------------------
        # Channels double and resolution halves at each stage after the first. That trade
        # keeps the compute per stage roughly constant while building up semantic depth.
        self.layer1 = self._make_layer(block, 64, num_blocks[0], stride=1)   # 32x32
        self.layer2 = self._make_layer(block, 128, num_blocks[1], stride=2)  # 16x16
        self.layer3 = self._make_layer(block, 256, num_blocks[2], stride=2)  # 8x8
        self.layer4 = self._make_layer(block, 512, num_blocks[3], stride=2)  # 4x4

        # --- classifier head ------------------------------------------------------------
        # Global average pooling collapses each 4x4 channel map to a single number, giving a
        # 512-vector per image. This is why the model is resolution-agnostic: any input size
        # ends up as 512 features here.
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(512 * block.expansion, num_classes)

        self._init_weights()

    def _make_layer(self, block, out_channels, num_blocks, stride):
        """Build one stage: `num_blocks` blocks, only the first of which may downsample."""
        # e.g. num_blocks=2, stride=2 -> strides [2, 1]: block 0 halves the map, block 1 keeps it.
        strides = [stride] + [1] * (num_blocks - 1)

        layers = []
        for s in strides:
            layers.append(block(self.in_channels, out_channels, s))
            # After the first block of a stage the channel count has changed, so subsequent
            # blocks in the same stage take the NEW count as their input.
            self.in_channels = out_channels * block.expansion

        return nn.Sequential(*layers)

    def _init_weights(self):
        """He/Kaiming initialisation — the right variance for ReLU networks."""
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                # fan_out mode keeps activation variance stable moving forward through layers.
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

        # "Zero-init residual": set the LAST BatchNorm gamma in each block to 0, so every block
        # starts out computing exactly F(x) = 0, i.e. the identity. The network begins as a
        # shallow one and grows into its depth. Worth ~0.5% accuracy and costs nothing.
        for m in self.modules():
            if isinstance(m, BasicBlock):
                nn.init.constant_(m.bn2.weight, 0)

    def forward(self, x):
        # x: (B, 3, 32, 32)
        out = F.relu(self.bn1(self.conv1(x)))   # (B, 64, 32, 32)
        out = self.layer1(out)                  # (B, 64, 32, 32)
        out = self.layer2(out)                  # (B, 128, 16, 16)
        out = self.layer3(out)                  # (B, 256, 8, 8)
        out = self.layer4(out)                  # (B, 512, 4, 4)
        out = self.avgpool(out)                 # (B, 512, 1, 1)
        out = torch.flatten(out, 1)             # (B, 512)
        return self.fc(out)                     # (B, 10) — raw LOGITS, no softmax


def ResNet18(num_classes=10):
    """ResNet-18: two BasicBlocks in each of the four stages."""
    return ResNet(BasicBlock, [2, 2, 2, 2], num_classes=num_classes)

## Stage 1 — architecture sanity check

No data, no training. Push a batch of random noise through the model and confirm the tensor
shapes are what the design says they should be. Runs in seconds on CPU, and catches the
overwhelming majority of architecture bugs (wrong strides, a mis-sized `fc` layer, a shortcut
that fails to line up) before any GPU time is spent.

In [ ]:
print("=" * 62)
print("STAGE 1 — architecture sanity check (random noise, no data)")
print("=" * 62)

model_check = ResNet18(num_classes=10)
model_check.eval()   # eval mode: BatchNorm uses running stats instead of batch stats

# A batch of 4 random "images" with CIFAR-10's shape: 3 colour channels, 32x32 pixels.
dummy = torch.randn(4, 3, 32, 32)

with torch.no_grad():             # no gradients needed for a shape check
    out = model_check(dummy)

print(f"input  shape: {tuple(dummy.shape)}")
print(f"output shape: {tuple(out.shape)}")

# The contract: one row per image, one logit per class.
assert out.shape == (4, 10), f"expected (4, 10), got {tuple(out.shape)}"

# --- trace the feature map through each stage -----------------------------------------
# Verifies the 32 -> 32 -> 16 -> 8 -> 4 downsampling actually happens as designed.
print("\nfeature map at each stage:")
with torch.no_grad():
    h = F.relu(model_check.bn1(model_check.conv1(dummy)))
    print(f"  stem   : {tuple(h.shape)}")
    for name in ("layer1", "layer2", "layer3", "layer4"):
        h = getattr(model_check, name)(h)
        print(f"  {name} : {tuple(h.shape)}")
    h = torch.flatten(model_check.avgpool(h), 1)
    print(f"  pooled : {tuple(h.shape)}")

# --- parameter count -------------------------------------------------------------------
n_params = sum(p.numel() for p in model_check.parameters() if p.requires_grad)
print(f"\ntrainable parameters: {n_params:,}")

# Reference ResNet-18 with a 10-class head is ~11.17M parameters. A big deviation means
# a stage has the wrong width or the wrong number of blocks.
assert 11_000_000 < n_params < 11_500_000, f"unexpected parameter count {n_params:,}"

# --- gradients actually flow ------------------------------------------------------------
# A forward pass can succeed while a detached branch silently blocks the backward pass.
# One backward step proves every parameter is genuinely connected to the loss.
model_check.train()
loss = F.cross_entropy(model_check(dummy), torch.randint(0, 10, (4,)))
loss.backward()
no_grad = [n for n, p in model_check.named_parameters() if p.requires_grad and p.grad is None]
assert not no_grad, f"these parameters received no gradient: {no_grad[:5]}"

print("\nSTAGE 1 PASSED — shapes, parameter count and gradient flow all correct.")
del model_check

## Data pipeline

**Augmentation (training set only).** CIFAR-10 has just 50,000 images and ResNet-18 has 11M
parameters, so it will memorise the training set unless we keep changing it:

- `RandomCrop(32, padding=4)` — pad to 40x40 then crop a random 32x32 window, so the subject
  shifts around the frame and the model cannot rely on absolute position.
- `RandomHorizontalFlip()` — a mirrored cat is still a cat. (Note this would be *wrong* for a
  dataset containing text or digits, where handedness carries meaning.)

The test set gets no augmentation — evaluation must be deterministic.

**Normalisation.** Per-channel mean/std of the CIFAR-10 training set, so each channel arrives
roughly zero-mean and unit-variance. The *same* constants are applied to test data; they are a
property of the training set, never recomputed on test.

**On the cluster:** compute nodes often have no outbound internet. Pre-download once on the
login node, then run jobs with `DAWN_DOWNLOAD=0` and `DAWN_DATA` pointing at that directory.

In [ ]:
# Per-channel statistics of the CIFAR-10 training set.
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD = (0.2470, 0.2435, 0.2616)

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),      # random translation
    transforms.RandomHorizontalFlip(),         # random mirror
    transforms.ToTensor(),                     # PIL/HWC uint8 [0,255] -> CHW float [0,1]
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

# Test: convert and normalise only. No randomness.
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

train_set = torchvision.datasets.CIFAR10(
    root=DATA_DIR, train=True, download=DOWNLOAD, transform=train_transform)
test_set = torchvision.datasets.CIFAR10(
    root=DATA_DIR, train=False, download=DOWNLOAD, transform=test_transform)

CLASSES = ("plane", "car", "bird", "cat", "deer",
           "dog", "frog", "horse", "ship", "truck")

print(f"train images: {len(train_set):,}")
print(f"test  images: {len(test_set):,}")
print(f"classes     : {len(CLASSES)} -> {CLASSES}")


def make_loaders(batch_size, train_subset=None, test_subset=None):
    """Build train/test DataLoaders, optionally restricted to the first N samples.

    `train_subset` / `test_subset` exist for the stage 2 smoke test, which needs only a
    couple of batches. Returning fresh loaders (rather than one global pair) keeps each
    stage independent and lets stage 4 use a different batch size if you want one.
    """
    tr = train_set if train_subset is None else Subset(train_set, range(train_subset))
    te = test_set if test_subset is None else Subset(test_set, range(test_subset))

    common = dict(
        num_workers=NUM_WORKERS,
        # pin_memory stages batches in page-locked host memory, making the host->GPU copy
        # faster and allowing it to overlap with compute. Pointless without a GPU.
        pin_memory=(DEVICE.type == "cuda"),
        # Keeping workers alive between epochs avoids re-spawning processes 30 times.
        persistent_workers=(NUM_WORKERS > 0),
    )

    train_loader = DataLoader(tr, batch_size=batch_size, shuffle=True,
                              drop_last=True, **common)
    test_loader = DataLoader(te, batch_size=batch_size, shuffle=False, **common)
    return train_loader, test_loader

## Training and evaluation functions

Both stage 3 and stage 4 call these, with mixed precision as a flag. Sharing one code path is
what makes the timing comparison meaningful — the only difference between the two runs is AMP.

### What mixed precision actually does

`autocast` runs the expensive ops (convolutions, matmuls) in **float16** while leaving
precision-sensitive ops (reductions, softmax, batchnorm statistics) in float32. On an A100 the
fp16 path uses the Tensor Cores, which are several times faster than the fp32 path, and halved
activation memory means larger batches fit.

### Why a GradScaler is required

fp16 has a much narrower exponent range than fp32. Small gradients simply become zero —
they *underflow* — and those weights then never update. The scaler multiplies the loss by a
large factor before `.backward()`, pushing gradients up into fp16's representable range, then
divides the gradients back down before the optimiser applies them. It also tracks overflows and
skips any step that produced inf/NaN, adapting the scale factor automatically.

With `enabled=False` both `autocast` and `GradScaler` become no-ops, so the FP32 run in stage 3
goes through exactly the same lines of code.

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, scheduler, scaler, use_amp):
    """Run one pass over `loader`. Returns (mean loss, accuracy) over the epoch."""
    model.train()   # BatchNorm uses batch statistics; dropout (if any) active

    running_loss, n_correct, n_seen = 0.0, 0, 0

    for xb, yb in loader:
        # non_blocking pairs with pin_memory to overlap the host->device copy with compute.
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True)

        # set_to_none=True releases the gradient tensors instead of filling them with zeros —
        # slightly faster and slightly lighter on memory than the default.
        optimizer.zero_grad(set_to_none=True)

        # Forward pass under autocast: ops that are safe in fp16 run in fp16.
        with torch.amp.autocast(DEVICE.type, enabled=use_amp):
            logits = model(xb)                 # (B, 10)
            loss = criterion(logits, yb)       # scalar

        # Scale the loss up, backpropagate, then unscale before stepping.
        scaler.scale(loss).backward()
        scaler.step(optimizer)                 # skipped automatically if grads overflowed
        scaler.update()                        # adapt the scale factor for next iteration

        # OneCycleLR steps per BATCH, not per epoch — the learning rate is a smooth curve
        # across the whole run rather than a staircase.
        scheduler.step()

        # --- bookkeeping (detached: no graph is retained) ---
        running_loss += loss.item() * yb.size(0)
        n_correct += (logits.argmax(dim=1) == yb).sum().item()
        n_seen += yb.size(0)

    return running_loss / n_seen, n_correct / n_seen


@torch.no_grad()   # disables autograd for the whole function: less memory, faster
def evaluate(model, loader, criterion, use_amp=False):
    """Evaluate on `loader`. Returns (mean loss, accuracy)."""
    model.eval()   # BatchNorm switches to running statistics — essential for correct results

    running_loss, n_correct, n_seen = 0.0, 0, 0

    for xb, yb in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True)

        with torch.amp.autocast(DEVICE.type, enabled=use_amp):
            logits = model(xb)
            loss = criterion(logits, yb)

        running_loss += loss.item() * yb.size(0)
        n_correct += (logits.argmax(dim=1) == yb).sum().item()
        n_seen += yb.size(0)

    return running_loss / n_seen, n_correct / n_seen


def build_optimizer(model, steps_per_epoch, epochs, max_lr):
    """SGD + OneCycleLR, the recipe behind the fast-training results.

    Why SGD and not Adam: for convnets on image classification, SGD with momentum and a
    well-scheduled learning rate generalises measurably better than Adam — typically 1-2%
    on CIFAR-10. Adam converges faster early but settles at a worse minimum.
    """
    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=max_lr,
        momentum=0.9,
        weight_decay=WEIGHT_DECAY,   # L2 regularisation, important with 11M params on 50k images
        nesterov=True,               # look-ahead momentum, small but free improvement
    )

    # OneCycle: warm up from max_lr/25 to max_lr over the first 25% of training, then anneal
    # down to near zero with a cosine curve. The large mid-training LR acts as a regulariser
    # (it refuses to settle into sharp minima); the low final LR lets it converge cleanly.
    # This is what lets 30 epochs do the work that a step schedule needs 100+ epochs for.
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=max_lr,
        steps_per_epoch=steps_per_epoch,
        epochs=epochs,
        pct_start=0.25,
        div_factor=25.0,        # initial lr = max_lr / 25
        final_div_factor=1e4,   # final lr = initial / 1e4
    )
    return optimizer, scheduler


def run_training(epochs, batch_size, max_lr, use_amp, tag, channels_last=False, verbose=True):
    """Train a fresh ResNet-18 end to end. Returns a dict of results and timings."""
    train_loader, test_loader = make_loaders(batch_size)

    model = ResNet18(num_classes=10).to(DEVICE)

    if channels_last:
        # NHWC memory layout. Tensor Core convolution kernels want this ordering; without it
        # cuDNN inserts transposes around every conv. Only helps on GPU, alongside AMP.
        model = model.to(memory_format=torch.channels_last)

    # label_smoothing spreads 10% of each target's probability mass over the other classes,
    # so the model is penalised for being over-confident. Reliably worth a few tenths of a
    # percent and makes training more stable at high learning rates.
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

    optimizer, scheduler = build_optimizer(model, len(train_loader), epochs, max_lr)

    # enabled=False makes every scaler call a pass-through, so the FP32 run uses this same path.
    scaler = torch.amp.GradScaler(DEVICE.type, enabled=use_amp)

    history = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": [], "epoch_time": []}

    # CUDA kernels are launched asynchronously, so wall-clock timing is only meaningful
    # if we synchronise first — otherwise we time the launch, not the work.
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    t_start = time.perf_counter()

    for epoch in range(1, epochs + 1):
        t_epoch = time.perf_counter()

        tr_loss, tr_acc = train_one_epoch(
            model, train_loader, criterion, optimizer, scheduler, scaler, use_amp)
        te_loss, te_acc = evaluate(model, test_loader, criterion, use_amp)

        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        dt = time.perf_counter() - t_epoch

        history["train_loss"].append(tr_loss)
        history["train_acc"].append(tr_acc)
        history["test_loss"].append(te_loss)
        history["test_acc"].append(te_acc)
        history["epoch_time"].append(dt)

        if verbose:
            # One line per epoch, flushed — readable in a SLURM log while the job runs.
            print(f"[{tag}] epoch {epoch:3d}/{epochs}  "
                  f"train_loss {tr_loss:.4f}  train_acc {tr_acc:.4f}  "
                  f"test_loss {te_loss:.4f}  test_acc {te_acc:.4f}  "
                  f"lr {scheduler.get_last_lr()[0]:.5f}  {dt:.1f}s", flush=True)

    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    total = time.perf_counter() - t_start

    result = {
        "tag": tag,
        "use_amp": use_amp,
        "epochs": epochs,
        "batch_size": batch_size,
        "max_lr": max_lr,
        "final_test_acc": history["test_acc"][-1],
        "best_test_acc": max(history["test_acc"]),
        "total_seconds": total,
        "mean_epoch_seconds": float(np.mean(history["epoch_time"])),
        "history": history,
    }

    print(f"\n[{tag}] finished in {total:.1f}s "
          f"({result['mean_epoch_seconds']:.1f}s/epoch)  "
          f"final {result['final_test_acc']:.4f}  best {result['best_test_acc']:.4f}")
    return model, result

## Stage 2 — tiny smoke test

Real CIFAR-10 data, but deliberately tiny: 2 batches for 2 epochs. This is not trying to learn
anything. It answers three questions cheaply, before committing to a long GPU queue:

1. Does the data pipeline actually deliver batches of the right shape and dtype?
2. Does the loss go *down*, i.e. is the optimiser wired up correctly?
3. Does anything produce NaN or inf?

A NaN here almost always means the learning rate is too high or a normalisation step is wrong.
Catching that in 30 seconds beats discovering it 20 minutes into a real run.

In [ ]:
print("=" * 62)
print("STAGE 2 — tiny smoke test (10 batches x 2 epochs)")
print("=" * 62)

SMOKE_BATCH = 64
SMOKE_BATCHES = 10
SMOKE_EPOCHS = 2

smoke_train, smoke_test = make_loaders(
    SMOKE_BATCH,
    train_subset=SMOKE_BATCH * SMOKE_BATCHES,   # 640 images
    test_subset=SMOKE_BATCH * SMOKE_BATCHES,
)

# Confirm the pipeline produces what the model expects before feeding it anything.
xb, yb = next(iter(smoke_train))
print(f"batch images: {tuple(xb.shape)}  dtype {xb.dtype}")
print(f"batch labels: {tuple(yb.shape)}  dtype {yb.dtype}  range [{yb.min()}, {yb.max()}]")
assert xb.shape[1:] == (3, 32, 32), f"unexpected image shape {tuple(xb.shape)}"
assert yb.dtype == torch.long, "labels must be int64 class indices for cross entropy"

smoke_model = ResNet18().to(DEVICE)
smoke_criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
smoke_opt, smoke_sched = build_optimizer(
    smoke_model, len(smoke_train), SMOKE_EPOCHS, max_lr=0.01)   # gentle LR for a 2-epoch run
smoke_scaler = torch.amp.GradScaler(DEVICE.type, enabled=False)

smoke_losses = []
for epoch in range(1, SMOKE_EPOCHS + 1):
    tr_loss, tr_acc = train_one_epoch(
        smoke_model, smoke_train, smoke_criterion, smoke_opt, smoke_sched,
        smoke_scaler, use_amp=False)
    smoke_losses.append(tr_loss)
    print(f"  epoch {epoch}: train_loss {tr_loss:.4f}  train_acc {tr_acc:.4f}")

# --- the three checks -------------------------------------------------------------------
assert all(np.isfinite(smoke_losses)), f"loss went non-finite: {smoke_losses}"
assert smoke_losses[-1] < smoke_losses[0], (
    f"loss did not decrease ({smoke_losses[0]:.4f} -> {smoke_losses[-1]:.4f}); "
    "check the optimiser, learning rate, and that zero_grad is being called")

te_loss, te_acc = evaluate(smoke_model, smoke_test, smoke_criterion)
assert np.isfinite(te_loss), "evaluation loss is not finite"
print(f"  eval : test_loss {te_loss:.4f}  test_acc {te_acc:.4f}")

print(f"\nSTAGE 2 PASSED — loss {smoke_losses[0]:.4f} -> {smoke_losses[-1]:.4f}, all finite.")
del smoke_model

## Stage 3 — full training run (FP32)

The real run: all 50,000 training images, `EPOCHS` epochs, standard float32.

**Gated** behind `DAWN_STAGE3=1` so stages 1-2 can be run on their own. Enable with:

```bash
DAWN_STAGE3=1 jupyter nbconvert --to notebook --execute part3_dawnbench.ipynb
```

Expect roughly 93-95% test accuracy after 30 epochs.

In [ ]:
print("=" * 62)
print("STAGE 3 — full FP32 training run")
print("=" * 62)

result_fp32 = None
if RUN_STAGE3:
    model_fp32, result_fp32 = run_training(
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        max_lr=MAX_LR,
        use_amp=False,
        tag="fp32",
    )
    torch.save(model_fp32.state_dict(), "resnet18_cifar10_fp32.pt")
    print("saved weights -> resnet18_cifar10_fp32.pt")

    target_met = result_fp32["best_test_acc"] > 0.90
    print(f"\n>90% accuracy target: {'MET' if target_met else 'NOT MET'} "
          f"({result_fp32['best_test_acc']:.4f})")
else:
    print("SKIPPED — set DAWN_STAGE3=1 to run this stage.")

## Stage 4 — mixed precision (the DAWNBench stretch goal)

Identical recipe to stage 3 — same epochs, same batch size, same learning rate — with mixed
precision and the channels-last memory layout switched on. Keeping everything else fixed is
deliberate: it means the timing difference is attributable to AMP, not to a confounded change
in the batch size or schedule.

On an A100, expect roughly **2-3x faster per epoch** at the same accuracy. 30 epochs should land
near the ~360 s / 94% DAWNBench-style target.

**Gated** behind `DAWN_STAGE4=1`.

> On the API naming: the lab sheet refers to `torch.cuda.amp`. That namespace still works but is
> deprecated as of PyTorch 2.x in favour of the device-generic `torch.amp.autocast("cuda", ...)`
> and `torch.amp.GradScaler("cuda", ...)`, which is what this notebook uses. Same mechanism.

In [ ]:
print("=" * 62)
print("STAGE 4 — mixed precision (AMP) training run")
print("=" * 62)

result_amp = None
if RUN_STAGE4:
    if DEVICE.type != "cuda":
        # autocast on CPU exists but uses bfloat16 and gives no speedup here; the comparison
        # would be meaningless, so be explicit rather than silently reporting a bad number.
        print("WARNING: no CUDA device — AMP gives no speedup on CPU. "
              "Run this stage on the GPU cluster.")

    model_amp, result_amp = run_training(
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        max_lr=MAX_LR,
        use_amp=True,
        tag="amp",
        channels_last=(DEVICE.type == "cuda"),
    )
    torch.save(model_amp.state_dict(), "resnet18_cifar10_amp.pt")
    print("saved weights -> resnet18_cifar10_amp.pt")

    acc_ok = result_amp["best_test_acc"] > 0.94
    time_ok = result_amp["total_seconds"] < 360
    print(f"\nDAWNBench stretch goal:")
    print(f"  94% accuracy : {'MET' if acc_ok else 'NOT MET'} ({result_amp['best_test_acc']:.4f})")
    print(f"  under 360 s  : {'MET' if time_ok else 'NOT MET'} ({result_amp['total_seconds']:.1f}s)")
else:
    print("SKIPPED — set DAWN_STAGE4=1 to run this stage.")

## Results — timing and accuracy comparison

Only produces output if at least one of stages 3 and 4 actually ran. Results are written to
`dawnbench_results.json` and the plots to `dawnbench_curves.png`, so a non-interactive SLURM
run leaves inspectable artifacts behind.

In [ ]:
results = [r for r in (result_fp32, result_amp) if r is not None]

if not results:
    print("No completed training runs — enable DAWN_STAGE3 and/or DAWN_STAGE4.")
else:
    print(f"{'run':<8}{'epochs':>8}{'batch':>8}{'s/epoch':>10}{'total s':>10}{'best acc':>11}")
    print("-" * 55)
    for r in results:
        print(f"{r['tag']:<8}{r['epochs']:>8}{r['batch_size']:>8}"
              f"{r['mean_epoch_seconds']:>10.1f}{r['total_seconds']:>10.1f}"
              f"{r['best_test_acc']:>11.4f}")

    if result_fp32 and result_amp:
        speedup = result_fp32["total_seconds"] / result_amp["total_seconds"]
        acc_delta = result_amp["best_test_acc"] - result_fp32["best_test_acc"]
        print("-" * 55)
        print(f"AMP speedup: {speedup:.2f}x     accuracy change: {acc_delta:+.4f}")

    # Persist the numbers (minus the full history) for the write-up.
    with open("dawnbench_results.json", "w") as f:
        json.dump([{k: v for k, v in r.items() if k != "history"} for r in results], f, indent=2)

    # --- curves ---------------------------------------------------------------------------
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    for r in results:
        ep = range(1, r["epochs"] + 1)
        ax1.plot(ep, r["history"]["train_loss"], label=f"{r['tag']} train")
        ax1.plot(ep, r["history"]["test_loss"], "--", label=f"{r['tag']} test")
        ax2.plot(ep, r["history"]["test_acc"], label=f"{r['tag']} test acc")

    ax1.set_xlabel("epoch"); ax1.set_ylabel("loss"); ax1.set_title("Loss")
    ax1.grid(True, alpha=0.3); ax1.legend()

    ax2.axhline(0.90, color="grey", ls=":", label="90% target")
    ax2.axhline(0.94, color="red", ls=":", label="94% stretch")
    ax2.set_xlabel("epoch"); ax2.set_ylabel("test accuracy"); ax2.set_title("Test accuracy")
    ax2.grid(True, alpha=0.3); ax2.legend()

    plt.tight_layout()
    plt.savefig("dawnbench_curves.png", dpi=120)
    print("\nsaved -> dawnbench_results.json, dawnbench_curves.png")
    plt.show()

## Per-class results and inference demo

Useful for the live demo: loads whichever model finished last and reports per-class accuracy
plus a handful of individual predictions.

In [ ]:
demo_model = None
if result_amp is not None:
    demo_model = model_amp
elif result_fp32 is not None:
    demo_model = model_fp32

if demo_model is None:
    print("No trained model in memory. Either run a training stage, or load saved weights:")
    print("  demo_model = ResNet18().to(DEVICE)")
    print("  demo_model.load_state_dict(torch.load('resnet18_cifar10_amp.pt'))")
else:
    _, demo_loader = make_loaders(256)

    demo_model.eval()
    n_correct = torch.zeros(10)
    n_total = torch.zeros(10)

    with torch.no_grad():
        for xb, yb in demo_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            preds = demo_model(xb).argmax(dim=1)
            for cls in range(10):
                mask = yb == cls
                n_total[cls] += mask.sum().item()
                n_correct[cls] += (preds[mask] == cls).sum().item()

    print(f"{'class':<8}{'accuracy':>10}{'n':>7}")
    print("-" * 25)
    for cls, name in enumerate(CLASSES):
        print(f"{name:<8}{(n_correct[cls] / n_total[cls]).item():>10.4f}{int(n_total[cls]):>7}")
    print("-" * 25)
    print(f"{'overall':<8}{(n_correct.sum() / n_total.sum()).item():>10.4f}"
          f"{int(n_total.sum()):>7}")

    # A few individual predictions, the kind of thing to show a demonstrator live.
    xb, yb = next(iter(demo_loader))
    with torch.no_grad():
        probs = F.softmax(demo_model(xb[:8].to(DEVICE)), dim=1).cpu()
    print("\nsample predictions:")
    for i in range(8):
        p, pred = probs[i].max(0)
        mark = "ok " if pred.item() == yb[i].item() else "XX "
        print(f"  {mark} true {CLASSES[yb[i]]:<7} pred {CLASSES[pred]:<7} conf {p:.3f}")

## Notes for the demo

Things a demonstrator is likely to probe, and the short answers:

**Why does the skip connection help?** Two reasons. It gives the block an easy way to represent
the identity function (drive `F(x)` to zero), so adding depth cannot make the achievable
training error worse. And the `+ x` path passes gradients backwards unattenuated, so early
layers still receive usable gradient signal.

**Why is the final ReLU after the addition?** So the identity path stays linear and unsquashed.
Putting a ReLU on the skip branch would clip negative values on every block and destroy exactly
the property the design is for.

**Why `bias=False` on the convs?** Every conv is followed by BatchNorm, which subtracts the
batch mean and adds its own learnable `beta`. A conv bias would be cancelled by the mean
subtraction — redundant parameters.

**Why the 3x3 stride-1 stem instead of the paper's 7x7 stride-2 + maxpool?** The paper targets
224x224 ImageNet. On a 32x32 CIFAR image that stem drops to 8x8 before the first residual
block, discarding most of the spatial information. The CIFAR variant keeps full resolution into
layer1.

**Why SGD rather than Adam?** For convnets on image classification, SGD + momentum with a good
schedule generalises better — typically 1-2% on CIFAR-10. Adam is faster to converge early but
settles into a worse minimum.

**What does the GradScaler do, and why is it needed?** fp16 has a narrow exponent range, so
small gradients underflow to zero and their weights stop updating. The scaler multiplies the
loss before backward to push gradients into fp16's range, unscales before the optimiser step,
and skips any step where an overflow produced inf/NaN.

**Why does OneCycleLR step per batch?** It defines the learning rate as a continuous curve over
total training *steps*, not epochs. Stepping per epoch would apply the schedule 1/391th as fast
and never complete the cycle.

**Why is train accuracy sometimes below test accuracy?** Training accuracy is measured on
augmented images (random crops and flips) while the model is still updating mid-epoch; test
accuracy is measured on clean images with a fixed model. Early on, the augmented task is
genuinely harder.

### Running the live epoch during the demo

```bash
DAWN_EPOCHS=1 DAWN_STAGE4=1 DAWN_DOWNLOAD=0 DAWN_DATA=$HOME/data \
  jupyter nbconvert --to notebook --execute --inplace part3_dawnbench.ipynb
```

One epoch takes roughly 10-20 s on an A100 and demonstrates the full pipeline end to end.